# Master Practicing Notebook: Number Theory & Applied Cryptography
**Student:** Mahfuzur Rahman  
**Student ID:** IT-23005  
**Course:** Cryptography and Cyber Law  
**Class Test:** CT-1 Practicing Number Theory  
**Repository:** [https://github.com/IT23005/Cryptography-and-Cyber-Law](https://github.com/IT23005/Cryptography-and-Cyber-Law)  
**Reference Workshop:** [IIT Bombay M&P Club - Number Theory & Cryptography](https://github.com/mnp-club/Number_Theory_and_Cryptography_Workshop)  

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/IT23005/Cryptography-and-Cyber-Law/blob/main/CT-1%20Practicing%20Number%20Theory/Number_Theory_and_Cryptography_Master.ipynb)

---
## Table of Contents
1. [Part I: Number Theory Foundations](#part-1)
2. [Part II: Classical and Modern Cryptosystems](#part-2)
3. [Part III: Practical RSA Cryptanalysis & The 7 Scenarios](#part-3)
4. [Part IV: The RSA Cryptographic Quest Resolution](#part-4)

---


# Number Theory Foundations for Cryptography
**Student:** Mahfuzur Rahman | **ID:** IT-23005  
**Course:** Cryptography and Cyber Law  
**Reference:** [IIT Bombay MNP Club - Number Theory & Cryptography Workshop](https://github.com/mnp-club/Number_Theory_and_Cryptography_Workshop)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/IT23005/Cryptography-and-Cyber-Law/blob/main/CT-1%20Practicing%20Number%20Theory/01_Number_Theory_Foundations.ipynb)

---
## Overview
This notebook implements and validates foundational number theory algorithms essential for modern public-key cryptosystems:
1. **Extended Euclidean Algorithm** and Bézout's Identity: $d = \gcd(a, b) = a \cdot x + b \cdot y$
2. **Least Common Multiple (LCM)** via $\gcd(a, b) \cdot \operatorname{lcm}(a, b) = a \cdot b$
3. **Linear Diophantine Equations**: $a \cdot x + b \cdot y = c$
4. **Modular Multiplicative Inverse and Modular Division**: $b / a \pmod n$
5. **Fast Modular Exponentiation (Square-and-Multiply)**: $b^e \pmod m$ in $O(\log e)$
6. **Chinese Remainder Theorem (CRT)** constructive algorithm
7. **Repunits, Perfect Numbers, and Fermat Pseudoprimes**


### 1. Extended Euclidean Algorithm
Computes $d = \gcd(a, b)$ and integer coefficients $(x, y)$ such that $d = ax + by$.


In [ ]:
def extended_gcd(a, b):
    """
    Extended Euclidean Algorithm.
    Requires: a >= b >= 0 and a + b > 0.
    Returns: (d, x, y) such that d = gcd(a, b) = a*x + b*y.
    """
    assert a >= b and b >= 0 and a + b > 0, "Invalid inputs: requires a >= b >= 0 and a+b > 0"
    
    if b == 0:
        return (a, 1, 0)
        
    d, x1, y1 = extended_gcd(b, a % b)
    x = y1
    y = x1 - (a // b) * y1
    
    assert a % d == 0 and b % d == 0
    assert d == a * x + b * y
    return (d, x, y)

# Verification tests
print("extended_gcd(8, 4)  =", extended_gcd(8, 4))
print("extended_gcd(27, 15) =", extended_gcd(27, 15))


### 2. Least Common Multiple (LCM)
Uses the identity $\gcd(a, b) \cdot \operatorname{lcm}(a, b) = a \cdot b$.


In [ ]:
def lcm(a, b):
    """Returns the LCM of two positive integers."""
    assert a > 0 and b > 0, "Inputs must be positive integers"
    g = extended_gcd(max(a, b), min(a, b))[0]
    return (a * b) // g

assert lcm(9, 3) == 9
assert lcm(17, 37) == 629
print("lcm(9, 3) =", lcm(9, 3))
print("lcm(17, 37) =", lcm(17, 37))


### 3. Linear Diophantine Equations
Solves $a \cdot x + b \cdot y = c$. An integer solution exists if and only if $\gcd(a, b) \mid c$.


In [ ]:
def diophantine(a, b, c):
    """
    Finds integer solution (x, y) to a*x + b*y = c.
    """
    swapped = False
    if a < b:
        a_orig, b_orig = a, b
        a_val, b_val = b, a
        swapped = True
    else:
        a_orig, b_orig = a, b
        a_val, b_val = a, b
        
    gcd_val, x0, y0 = extended_gcd(a_val, b_val)
    assert c % gcd_val == 0, f"No solution: gcd({a_orig}, {b_orig}) does not divide {c}"
    
    factor = c // gcd_val
    x0 *= factor
    y0 *= factor
    
    x, y = (y0, x0) if swapped else (x0, y0)
    assert a_orig * x + b_orig * y == c
    return (x, y)

print("diophantine(34, 51, 170) =", diophantine(34, 51, 170))
print("diophantine(91, 117, 65)  =", diophantine(91, 117, 65))


### 4. Modular Multiplicative Inverse and Modular Division
Calculates $a^{-1} \pmod n$ such that $a \cdot a^{-1} \equiv 1 \pmod n$, and computes $b / a \pmod n$.


In [ ]:
def modInverse(a, n):
    """Calculates multiplicative inverse of a modulo n using extended Euclid."""
    assert n > 1
    a_reduced = a % n
    assert a_reduced > 0
    if a_reduced >= n:
        d, x, y = extended_gcd(a_reduced, n)
        inv = x % n
    else:
        d, x, y = extended_gcd(n, a_reduced)
        inv = y % n
    assert d == 1, f"Modular inverse does not exist: gcd({a}, {n}) != 1"
    return (inv + n) % n

def divide(b, a, n):
    """Returns x in [0, n-1] such that a * x = b (mod n)."""
    assert n > 1 and a > 0
    a_inv = modInverse(a, n)
    assert (a * a_inv) % n == 1
    return (a_inv * b) % n

print("divide(9, 5, 3)     =", divide(9, 5, 3))
print("divide(223, 67, 54) =", divide(223, 67, 54))


### 5. Fast Modular Exponentiation (Square-and-Multiply)
Computes $b^e \pmod m$ in $O(\log_2 e)$ modular multiplications, preventing integer overflow.


In [ ]:
def FastModularExponentiation(b, e, m):
    """Computes b^e mod m using Square-and-Multiply algorithm."""
    result = 1
    base = b % m
    exp = e
    while exp > 0:
        if exp % 2 == 1:
            result = (result * base) % m
        base = (base * base) % m
        exp //= 2
    return result

print("FastModularExponentiation(4, 5, 3) =", FastModularExponentiation(4, 5, 3))
print("Built-in pow(4, 5, 3)              =", pow(4, 5, 3))


### 6. Chinese Remainder Theorem (CRT)
Constructs the unique integer $r \pmod{n_1 n_2}$ satisfying:
$$r \equiv r_1 \pmod{n_1} \quad 	ext{and} \quad r \equiv r_2 \pmod{n_2}$$
where $\gcd(n_1, n_2) = 1$.


In [ ]:
def ChineseRemainderTheorem(n1, r1, n2, r2):
    """Constructs solution to system of congruences modulo n1*n2."""
    d, x, y = extended_gcd(n1, n2) if n1 >= n2 else extended_gcd(n2, n1)
    assert d == 1, "Moduli must be coprime"
    
    N = n1 * n2
    if n1 >= n2:
        # n1*x + n2*y = 1 => n2*y = 1 mod n1, n1*x = 1 mod n2
        r = (r1 * n2 * y + r2 * n1 * x) % N
    else:
        # n2*x + n1*y = 1 => n1*y = 1 mod n2, n2*x = 1 mod n1
        r = (r2 * n1 * y + r1 * n2 * x) % N
        
    r = (r + N) % N
    assert r % n1 == r1
    assert r % n2 == r2
    return r

print("ChineseRemainderTheorem(13, 7, 6, 5)  =", ChineseRemainderTheorem(13, 7, 6, 5))
print("ChineseRemainderTheorem(28, 20, 19, 3) =", ChineseRemainderTheorem(28, 20, 19, 3))


### 7. Repunits, Even Perfect Numbers & Fermat Pseudoprimes
Exploration of advanced theoretical properties from Day 2 of the workshop.


In [ ]:
# Repunits: R_n = 111...1 (n digits)
def repunit(n):
    return (10**n - 1) // 9

print("Repunits R_1 to R_6:", [repunit(i) for i in range(1, 7)])

# Even Perfect Numbers & Euclid-Euler Theorem
# n = 2^(p-1) * (2^p - 1) where (2^p - 1) is prime
def check_even_perfect(n):
    temp = n
    k = 0
    while temp % 2 == 0:
        temp //= 2
        k += 1
    p = k + 1
    m = (1 << p) - 1
    if temp == m:
        # check primality of m
        for i in range(2, int(m**0.5) + 1):
            if m % i == 0:
                return False, None
        return True, p
    return False, None

for num in [6, 28, 496, 8128]:
    is_p, p_val = check_even_perfect(num)
    print(f"Number {num}: Even Perfect = {is_p} (Mersenne prime exponent p = {p_val})")

# Fermat Pseudoprimes to base 2
# Composite n such that 2^(n-1) = 1 (mod n)
def is_pseudoprime_base2(n):
    if n < 4: return False
    # Check composite
    composite = any(n % i == 0 for i in range(2, int(n**0.5) + 1))
    if not composite: return False
    return FastModularExponentiation(2, n - 1, n) == 1

print("Is 341 a Fermat pseudoprime base 2?", is_pseudoprime_base2(341))
print("Factors of 341:", 11, "x", 31)
print("Is 561 a Carmichael number (pseudoprime to all coprime bases)?", is_pseudoprime_base2(561))


### 1. One-Time Pad & String Conversions
Text to hex conversion, bitwise XOR, and perfect encryption.


In [ ]:
from functools import reduce

def toHex(s):
    return ''.join(f"{ord(c):02x}" for c in s)

def toStr(hex_str):
    if len(hex_str) % 2 != 0:
        hex_str = '0' + hex_str
    return bytes.fromhex(hex_str).decode('latin1')

def Xor(hex1, hex2):
    assert len(hex1) == len(hex2)
    b1 = bytes.fromhex(hex1)
    b2 = bytes.fromhex(hex2)
    return bytes(x ^ y for x, y in zip(b1, b2)).hex()

message = "secret message"
key     = "my secret keys"
c = Xor(toHex(message), toHex(key))
print("Ciphertext hex:", c)
decrypted = toStr(Xor(c, toHex(key)))
print("Decrypted msg :", decrypted)
assert decrypted == message


### 2. Many-Time Pad (Two-Time Pad Attack)
When key $K$ is re-used to encrypt two messages $M_1$ and $M_2$:
$$C_1 = M_1 \oplus K, \quad C_2 = M_2 \oplus K \implies C_1 \oplus C_2 = M_1 \oplus M_2$$
The key cancels out! An eavesdropper can use **crib dragging** to slide guessed English words across the XOR stream.


In [ ]:
message1 = "steal the secret"
message2 = "the cat the bird"
key      = "supersecretverys"

ciphertext1 = Xor(toHex(message1), toHex(key))
ciphertext2 = Xor(toHex(message2), toHex(key))

xor_ciphertexts = Xor(ciphertext1, ciphertext2)
xor_messages    = Xor(toHex(message1), toHex(message2))
assert xor_ciphertexts == xor_messages
print("XOR of ciphertexts equals XOR of messages:", xor_ciphertexts == xor_messages)

def TryGuessingSubstring(substring, message_length, xor_messages_hex):
    good_guesses = []
    sub_len = len(substring)
    for pos in range(message_length - sub_len + 1):
        padded_guess = chr(0) * pos + substring + chr(0) * (message_length - sub_len - pos)
        guess_hex = toHex(padded_guess)
        other_part = toStr(Xor(guess_hex, xor_messages_hex))[pos:pos + sub_len]
        if all(ch.isalpha() or ch.isspace() for ch in other_part):
            good_guesses.append((pos, substring, other_part))
    return good_guesses

guesses = TryGuessingSubstring(" the ", len(message1), xor_ciphertexts)
print("
Crib Dragging Results for ' the ':")
for pos, crib, other in guesses:
    print(f"  Position {pos}: One message has '{crib}', other message has '{other}'")


### 3. Diffie-Hellman Key Exchange Protocol
Simulation of Alice and Bob establishing a shared secret over an insecure channel.


In [ ]:
N = 11    # Public Prime Modulus
g = 7     # Public Generator (Primitive Root modulo 11)

# Alice and Bob pick private keys
a = 2     # Alice's private key
b = 4     # Bob's private key

# Public values transmitted over network
A = pow(g, a, N)    # Alice transmits A
B = pow(g, b, N)    # Bob transmits B

print("Alice transmits A =", A)
print("Bob transmits B   =", B)

# Each derives the shared secret
K_alice = pow(B, a, N)
K_bob   = pow(A, b, N)

assert K_alice == K_bob
print("Shared Synthesized Key =", K_alice)
print("Why it works: (g^b)^a = (g^a)^b = g^(ab) mod N")


### 4. RSA Cryptosystem Implementation
Full pipeline: Text encoding to large integer, Key generation, Encryption, Decryption, and Totient Verification.


In [ ]:
def ConvertToInt(s):
    res = 0
    for ch in s: res = res * 256 + ord(ch)
    return res

def ConvertToStr(n):
    res = []
    while n > 0:
        res.append(chr(n % 256))
        n //= 256
    return "".join(reversed(res))

def ExtendedEuclid(a, b):
    if b == 0: return (1, 0)
    (x, y) = ExtendedEuclid(b, a % b)
    k = a // b
    return (y, x - k * y)

def InvertModulo(a, n):
    (b, x) = ExtendedEuclid(a, n)
    if b < 0: b = (b % n + n) % n
    return b

p = 1000000000039
q = 1000000000061
N = p * q
phiN = (p - 1) * (q - 1)
e = 65537
d = InvertModulo(e, phiN)

print("Modulus N :", N)
print("Public Key (e, N) :", (e, N))
print("Private Key (d, N):", (d, N))

msg = "attack"
m_int = ConvertToInt(msg)
c_int = pow(m_int, e, N)
decrypted_int = pow(c_int, d, N)
decrypted_msg = ConvertToStr(decrypted_int)

print("
Plaintext :", msg)
print("Ciphertext:", c_int)
print("Decrypted :", decrypted_msg)
assert decrypted_msg == msg


### 5. Shor's Quantum Factoring Algorithm Simulation
Classical reduction: How quantum period-finding factors $N = p \cdot q$ using $\gcd(a^{r/2} \pm 1, N)$.


In [ ]:
import math

N = 35
a = 8  # gcd(8, 35) = 1

# Finding period r such that a^r = 1 mod N
# On a quantum computer, this is performed in O((log N)^2) using Quantum Fourier Transform
r = 1
while pow(a, r, N) != 1:
    r += 1

print(f"Period r of {a} modulo {N} is: {r}")
assert r % 2 == 0, "Period must be even"
half_val = pow(a, r // 2, N)
assert half_val != N - 1, "Trivial factor condition"

# Factors extracted via GCD:
factor1 = math.gcd(half_val - 1, N)
factor2 = math.gcd(half_val + 1, N)

print(f"Extracted Factor 1: gcd({half_val} - 1, {N}) = {factor1}")
print(f"Extracted Factor 2: gcd({half_val} + 1, {N}) = {factor2}")
assert factor1 * factor2 == N
print(f"Successfully factored {N} into {factor1} x {factor2}!")


### Core Helpers
String encoding, decoding, iterative modular exponentiation, and Extended Euclidean inversion.


In [ ]:
def ConvertToInt(s):
    res = 0
    for ch in s: res = res * 256 + ord(ch)
    return res

def ConvertToStr(n):
    res = []
    while n > 0:
        res.append(chr(n % 256))
        n //= 256
    return "".join(reversed(res))

def PowMod(a, n, mod):
    res = 1
    base = a % mod
    exp = n
    while exp > 0:
        if exp % 2 == 1:
            res = (res * base) % mod
        base = (base * base) % mod
        exp //= 2
    return res

def ExtendedEuclid(a, b):
    if b == 0: return (1, 0)
    (x, y) = ExtendedEuclid(b, a % b)
    k = a // b
    return (y, x - k * y)

def InvertModulo(a, n):
    (b, x) = ExtendedEuclid(a, n)
    if b < 0: b = (b % n + n) % n
    return b

def IntSqrt(n):
    if n <= 0: return 0
    low, high, ans = 1, n, 1
    while low <= high:
        mid = (low + high) // 2
        if mid * mid <= n:
            ans = mid
            low = mid + 1
        else:
            high = mid - 1
    return ans


### Question 1: RSA Encryption
Computes ciphertext $C \equiv M^e \pmod N$.


In [ ]:
def Encrypt(message, modulo, exponent):
    m_int = ConvertToInt(message)
    return PowMod(m_int, exponent, modulo)

print("Encrypted 'attack' with e=23917:", Encrypt("attack", 1000000007 * 1000000009, 23917))


### Question 2: RSA Decryption
Recovers $M \equiv C^d \pmod N$ using private key components $(p, q, e)$.


In [ ]:
def Decrypt(ciphertext, p, q, exponent):
    phiN = (p - 1) * (q - 1)
    d = InvertModulo(exponent, phiN)
    modulo = p * q
    m_int = PowMod(ciphertext, d, modulo)
    return ConvertToStr(m_int)

p = 1000000007
q = 1000000009
exponent = 23917
c = Encrypt("attack", p * q, exponent)
recovered = Decrypt(c, p, q, exponent)
print("Decrypted message:", recovered)
assert recovered == "attack" 


### Question 3: Attack 1 - Known-Plaintext / Dictionary Attack (`DecipherSimple`)
When messages come from a small candidate pool (e.g., Alice sending military orders), an attacker encrypts each possibility.


In [ ]:
def DecipherSimple(ciphertext, modulo, exponent, potential_messages):
    for candidate in potential_messages:
        if Encrypt(candidate, modulo, exponent) == ciphertext:
            return candidate
    return "don't know"

modulo = 1000000007 * 1000000009
exponent = 65537
c = Encrypt("attack", modulo, exponent)
detected = DecipherSimple(c, modulo, exponent, ["attack", "don't attack", "wait"])
print("Deciphered message from dictionary:", detected)
assert detected == "attack" 


### Question 4: Attack 2 - Small Prime Factorization (`DecipherSmallPrime`)
If one prime factor is small ($p < 1,000,000$), trial division breaks the modulus instantly.


In [ ]:
def DecipherSmallPrime(ciphertext, modulo, exponent):
    found_p = None
    if modulo % 2 == 0:
        found_p = 2
    else:
        for i in range(3, 1000000, 2):
            if modulo % i == 0:
                found_p = i
                break
    assert found_p is not None, "Prime not found under 1,000,000"
    q = modulo // found_p
    return Decrypt(ciphertext, found_p, q, exponent)

# Test with p < 1,000,000
p4 = 999983
q4 = 10000000000000000051
n4 = p4 * q4
e4 = 65537
c4 = Encrypt("attack", n4, e4)
print("Decrypted small prime ciphertext:", DecipherSmallPrime(c4, n4, e4))


### Question 5: Attack 3 - Fermat's Factorization for Close Primes (`DecipherSmallDiff`)
When $|p - q| < 5000$, $N = a^2 - b^2$ where $a pprox \lceil \sqrt{N} ceil$ and $b = |p - q| / 2$.


In [ ]:
def DecipherSmallDiff(ciphertext, modulo, exponent):
    a = IntSqrt(modulo)
    if a * a < modulo:
        a += 1
    for _ in range(50000):
        t2 = a * a - modulo
        b = IntSqrt(t2)
        if b * b == t2:
            p = a - b
            q = a + b
            return Decrypt(ciphertext, p, q, exponent)
        a += 1
    raise ValueError("Factors not found")

p5 = 1000000007
q5 = 1000000009
n5 = p5 * q5
e5 = 239
c5 = Encrypt("attack", n5, e5)
print("Decrypted small diff ciphertext:", DecipherSmallDiff(c5, n5, e5))


### Question 6: Attack 4 - Shared Prime Factor Attack (`DecipherCommonDivisor`)
If two keys share a prime factor $p = \gcd(N_1, N_2)$, Euclid's algorithm factors both in $O(\log N)$.


In [ ]:
def GCD(a, b):
    while b: a, b = b, a % b
    return a

def DecipherCommonDivisor(c1, n1, e1, c2, n2, e2):
    p = GCD(n1, n2)
    assert p > 1
    q1 = n1 // p
    q2 = n2 // p
    return (Decrypt(c1, p, q1, e1), Decrypt(c2, p, q2, e2))

p6 = 101
q1_6 = 18298970732541109011012304219376080251334480295537316123696052970419466495220522723330315111017831737980079504337868198011077274303193766040393009648852841770668239779097280026631944319501437547002412556176186750790476901358334138818777298389724049250700606462316428106882097210008142941838672676714188593227684360287806974345181893018133710957167334490627178666071809992955566020058374505477745993383434501768887090900283569055646901291270870833498474402084748161755197005050874785474707550376333429671113753137201128897550014524209754619355308207537703754006699795711188492048286436285518105948050401762394690148387
q2_6 = 10000000000000000051

n1 = p6 * q1_6
n2 = p6 * q2_6
c1 = Encrypt("attack", n1, 239)
c2 = Encrypt("wait", n2, 17)
print("Decrypted shared divisor messages:", DecipherCommonDivisor(c1, n1, 239, c2, n2, 17))


### Question 7: Attack 5 - Håstad's Broadcast Attack (`DecipherHastad`)
When the same message $M$ is broadcast with $e=2$ to 2 recipients, Chinese Remainder Theorem reconstructs $M^2$ over the integers, which is inverted by computing $\lfloor \sqrt{M^2} floor$.


In [ ]:
def ChineseRemainderTheorem(n1, r1, n2, r2):
    (x, y) = ExtendedEuclid(n1, n2)
    N = n1 * n2
    r = (r1 * n2 * y + r2 * n1 * x) % N
    return (r + N) % N

def DecipherHastad(first_ciphertext, first_modulo, second_ciphertext, second_modulo):
    m2 = ChineseRemainderTheorem(first_modulo, first_ciphertext, second_modulo, second_ciphertext)
    m = IntSqrt(m2)
    assert m * m == m2, "Not an exact square"
    return ConvertToStr(m)

p1 = 790383132652258876190399065097
q1 = 662503581792812531719955475509
p2 = 656917682542437675078478868539
q2 = 1263581691331332127259083713503
n1 = p1 * q1
n2 = p2 * q2

c1 = Encrypt("attack", n1, 2)
c2 = Encrypt("attack", n2, 2)
print("Hastad attack recovered message:", DecipherHastad(c1, n1, c2, n2))


### Stage 1: Puzzles 1 & 2
Unlocking the First Secret Link.


In [ ]:
from cryptosystems import ConvertToInt, ConvertToStr, FastModularExponentiation, InvertModulo
from rsa_attacks import DecipherSimple

def Decrypt(ciphertext, p, q, exponent):
    phiN = (p - 1) * (q - 1)
    d = InvertModulo(exponent, phiN)
    modulo = p * q
    m_int = FastModularExponentiation(ciphertext, d, modulo)
    return ConvertToStr(m_int)

# Puzzle 1
p1 = 779849711281
q1 = 748173698927
e1 = 1018651
c1 = 148784435264686331994392
decrypt_first_puzzle = Decrypt(c1, p1, q1, e1)
print("Puzzle 1 Decryption:", decrypt_first_puzzle)

# Puzzle 2
c2 = 336184023047118677086739
m2 = 1110014195838866450995043
e2 = 767549
potential_messages = ["http://goo.gl/", "http://bit.ly/", "http://tinyurl.com/", "http://t.co/", "http://ow.ly/", "https://is.gd/", "https://buff.ly/", "http://adf.ly/", "http://bit.do/"]
decrypt_second_puzzle = DecipherSimple(c2, m2, e2, potential_messages)
print("Puzzle 2 Decryption:", decrypt_second_puzzle)

secret_link_1 = decrypt_second_puzzle + decrypt_first_puzzle
print("
>>> SECRET LINK 1:", secret_link_1)


### Stage 2: Puzzles 3 & 4
Unlocking the Second Secret Link using values retrieved from Document 1.


In [ ]:
from rsa_attacks import DecipherSmallPrime, DecipherSmallDiff

# Values from Document 1
c3 = 2275574988111277110437311214647848935000427550724521671610776681372035260669927879546598212402059626568529572296378786125308417822064565716822176055495534704248368924219660507214955218285229437531331022539446136060045737304497347919927268903882567968402406701512120140117419155759639779776379512631504909857
m3 = 4517997441484302732739462818197633745340869919448215356148620255867108411028738486895653782573554891335084161684285442351914740415677262295900238601126392055761648706990593425725625329173683080677363007706828763867611649178136159927562319810830366004181626803874119100160889961902072311468445073289996506931
e3 = 1927740953777951565602110467665746890410015037617105427402060961409149865403375973001399584612131360178672080227818127622463053259278787951537329431243

decrypt_third_puzzle = DecipherSmallPrime(c3, m3, e3)
print("Puzzle 3 Decryption:", decrypt_third_puzzle)

# Values for Puzzle 4 (Fermat Factorization)
c4 = 17250119573232938159495151836907345788113398812430175654066191733201327934519622049877939234487323106446778939262441322441499883118016470240906983286600405405975275723333283015774857915215814576298063848917112178624770247286048114974808099004729887688547429197942578127510912101233759373809217128865040107305284699411659572170044538767976158851028945020639377891878532187396172245359826784724912031113990729909962190626715090093246089745719249482899605241682396631713155172212615370473237731019162671529937751370463768746592055682933647328743947355593762481102849672215738899019205704001633289554719966
m4 = 51719336593283668370261446912126485319255258778617224708410520647959236475322767644636584229070380964088870777607922268179572997129414217601241230215567764098337343256793524076040868978141016236299799031426456716364496058717536559180352016088612210860181989844954906788705998058534134464446292058222426189127398972577715363401457729737792004774929189832678271680193791993183578966529414824171500027009334274604018078392873654591487139190491950796770048687394408709037045383279375259210879772320878973829171531428762995383292175994184457145115155299673171196041635346062296031613203020122378411630001719
e4 = 2370692877507893030572035746060036014373915956107619689350920177346027898716147057536168003507966056257828756067470679338655825857154534154205294932307

decrypt_fourth_puzzle = DecipherSmallDiff(c4, m4, e4)
print("Puzzle 4 Decryption:", decrypt_fourth_puzzle)

secret_link_2 = decrypt_third_puzzle + decrypt_fourth_puzzle
print("
>>> SECRET LINK 2:", secret_link_2)


### Stage 3: Puzzles 6 & 7 and the Final Answer
Solving the final challenges using parameters from Document 2.


In [ ]:
from rsa_attacks import DecipherCommonDivisor, DecipherHastad

# Document 2: Common divisor parameters
c1_6 = 23402197839825692972957660045778005842149570897525565884030609841447497251897989699441878864356827076382914144503306327825195586612359555424774490982037582237066994974597680950158893991141848342087005739774563510316219462834998856466395854239468748012474813939969736279381474141408975441930226654470509840545902057273419529586319420996953635246047632517133343837614016964611767487664771097138242956384358875235954455242525595951902976992404807998173614628224868047333599482826829214633744976147539512358779840250950906473302037621404671062661398282452008807449216135596254412709706306612180825630285207
m1_6 = 58430823052528382232133162302686450950325469505875659240893229402912849840784309398807910065914827693927654379087691699343872463431863820528404206238621375034134223224550898236635277014485802812169842923892545064506382750914506131733421755620169771252068722149093463194110389888198706547399113274669324939230289590384817282220066815621515694361776987606124224166139632882923972455465687163365892873407377154186582920232983732264041961425519725867802358413817383859468852644174951734478657652205963797131615641353571742901542970047092056441684257125633460892715564219695534982132772847099500990153159493
e1_6 = 2746118316073165006639038412489080229254239843347153634155286882078957003180280976570735902213132927273096496684341374829942796710900231180504644399747

c2_6 = 3072704045398319834608585347909200712117029834128064442920028275297571725588844950848455160588121610317370817106044792955922551008270509007724543503222639431533406555370648161391360353447039467043718494772215515311420926819323504214073692302359076868569278351487334040235577573584610975969774812023377781048195207371503287730490209750445981869076182666622029473694669920172468716779937348673868171294362620903115903645873289564367176851434972937848591902468604400091051094342304122317377572964272370825515688006084797380707045470951321039467067019460709497651682710286878824565888446939229639383746
m2_6 = 42955660491543430460148980223483714568674234383080724206879432109744068992302351872678094322626198782875395264399683702531442650135501135314224418412956329257244753538669224867463416933578848236691757751760346388773871177654731330626106150400017398309330888903195999995955385444097920846918948184509773583149893677181571427932535786503554494314244294387551141996689565647925375574586590998569956897670722964200467030891957754959187718494336504559943885199686487499306417596091732019643533054349876912582987170110689841980733691588287144182824378328622431439004258091442919814253322774125364502939460173
e2_6 = 1696307946163562648509752484630349193234799326977939859599277182356207670242676510190455023892125378267322231053560789876259990406520567562708111385869

decrypt_sixth_puzzle = DecipherCommonDivisor(c1_6, m1_6, e1_6, c2_6, m2_6, e2_6)
print("Puzzle 6 Part 1:", decrypt_sixth_puzzle[0])
print("Puzzle 6 Part 2:", decrypt_sixth_puzzle[1])

# Document 2: Hastad broadcast attack parameters
c1_h = 1695925818934701958536732606471007147871697773728881305981426383004125888187531096540654444788889942827682032651915812560
m1_h = 2019703097262944143745237345710626601616589344196612447233041278945420303895066774752152001687355425741826173570952817341

c2_h = 438180481071044087096048173519586369091293503891810459558098521655659500752405553498472135529822589328291372879622802017
m2_h = 968914506353797853274340817936111164236562754898105214422095149641604042531040243209810736059746240506447210425767208577

decrypt_seventh_puzzle = DecipherHastad(c1_h, m1_h, c2_h, m2_h)
print("Puzzle 7 Part 3:", decrypt_seventh_puzzle)

# Complete Quest Answer
final_answer = decrypt_sixth_puzzle[0] + decrypt_sixth_puzzle[1] + decrypt_seventh_puzzle
print("
" + "=" * 80)
print("THE FINAL ANSWER TO RSA QUEST:")
print(f'"{final_answer}"')
print("=" * 80)
